# Rate Limiting Algorithms — Protecting Services from Overload & Abuse

## 🧠 Mental Model

> **A rate limiter is a pipe with a maximum flow rate. Too much pressure and
> excess requests are rejected at the gate before they reach the service —
> protecting downstream systems from overload.**

### Four Algorithms — Trade-offs at a Glance

| Algorithm | Memory | Burst tolerance | Boundary burst? | Best for |
|---|---|---|---|---|
| Fixed Window | O(1) | Up to 2× limit | Yes (double-burst at edge) | Simple APIs, internal services |
| Sliding Window Log | O(requests) | None — exact | No | Strict per-user billing |
| Token Bucket | O(1) | Yes (bucket capacity) | Natural burst allowance | External APIs (GitHub, Stripe) |
| Leaky Bucket | O(queue) | Absorbs, queues | None — smooth output | Video streaming, constant-rate consumers |

### The Fixed Window Edge Problem

```
Window 1 [0s—60s]    Window 2 [60s—120s]
  ←── limit=10 ──→    ←── limit=10 ──→
           ↑t=59s: 10 req    t=61s: 10 req↑
           Both windows see 10 (within limit)
           But the service got 20 req in 2 seconds!
```
→ Use sliding window or token bucket when edge bursts are unacceptable.

### Token Bucket vs Leaky Bucket

```
TOKEN BUCKET                    LEAKY BUCKET
─────────────                   ─────────────
Tokens refill at R/sec          Requests drain at R/sec
Bucket capacity = B tokens      Queue capacity = Q requests
Client CAN BURST up to B req    Output is PERFECTLY SMOOTH at rate R
→ API clients that legitimately  → Downstream needing constant input
  need to batch requests           (billing, video, telemetry pipelines)
```

### Distributed Rate Limiting Gotcha

```
10 instances × 100 req/sec per instance = 1,000 req/sec total
A single user hitting one instance: limited to 100 ✓
But that same user can hit ALL 10 instances: 1,000 req/sec ✗

FIX: Shared counter in Redis
  Redis INCR key EX 60          ← atomic fixed window
  Redis ZADD + ZCOUNT + EXPIRE  ← sliding window log
  Use Lua scripts for atomicity!
```

### 🌍 Where Rate Limiting is Used in Production

| System | Algorithm | Limit |
|---|---|---|
| GitHub API | Fixed window | 5,000 req/hour per token |
| Stripe API | Token bucket | Per-endpoint, per-API-key |
| AWS API Gateway | Token bucket | Configurable per route |
| Nginx `limit_req` | Leaky bucket | `burst=5 nodelay` |
| Cloudflare | Sliding window | Distributed via Redis |

---
## Implementation

In [ ]:
"""
04 — System Design: Rate Limiting Algorithms
============================================

Runnable companion to PDF Book V "Protecting a service from overload & abuse".

A rate limiter caps how many requests a client may make per unit time. The four
classic algorithms trade smoothness, burst tolerance, and memory:

  * FIXED WINDOW   — count per calendar window (simple; allows 2x burst at edges)
  * SLIDING WINDOW LOG — timestamps in a rolling window (exact; more memory)
  * TOKEN BUCKET   — refill tokens at a steady rate; allows controlled bursts
  * LEAKY BUCKET   — requests drain at a fixed rate (smooths output)

A deterministic fake clock makes the behavior testable without real sleeping.
"""

from __future__ import annotations

from collections import deque

In [ ]:
class Clock:
    """Injectable fake time so tests are deterministic (no real sleeping)."""
    def __init__(self, t: float = 0.0):
        self.t = t

    def tick(self, seconds: float) -> None:
        self.t += seconds

    def now(self) -> float:
        return self.t

In [ ]:
class FixedWindow:
    def __init__(self, limit: int, window: float, clock: Clock):
        self._limit, self._window, self._clock = limit, window, clock
        self._count = 0
        self._start = clock.now()

    def allow(self) -> bool:
        now = self._clock.now()
        if now - self._start >= self._window:   # new window -> reset
            self._start = now
            self._count = 0
        if self._count < self._limit:
            self._count += 1
            return True
        return False

In [ ]:
class SlidingWindowLog:
    def __init__(self, limit: int, window: float, clock: Clock):
        self._limit, self._window, self._clock = limit, window, clock
        self._hits: deque[float] = deque()

    def allow(self) -> bool:
        now = self._clock.now()
        while self._hits and now - self._hits[0] >= self._window:
            self._hits.popleft()                # drop timestamps older than window
        if len(self._hits) < self._limit:
            self._hits.append(now)
            return True
        return False

In [ ]:
class TokenBucket:
    def __init__(self, capacity: int, refill_per_sec: float, clock: Clock):
        self._cap = capacity
        self._rate = refill_per_sec
        self._clock = clock
        self._tokens = float(capacity)
        self._last = clock.now()

    def allow(self, cost: float = 1.0) -> bool:
        now = self._clock.now()
        self._tokens = min(self._cap, self._tokens + (now - self._last) * self._rate)
        self._last = now
        if self._tokens >= cost:
            self._tokens -= cost
            return True
        return False

In [ ]:
def demo() -> None:
    # Fixed window: 3 per second, 4th denied; resets next window.
    clk = Clock()
    fw = FixedWindow(limit=3, window=1.0, clock=clk)
    assert [fw.allow() for _ in range(4)] == [True, True, True, False]
    clk.tick(1.0)
    assert fw.allow() is True                    # new window
    print("   fixed window: 3 allowed, 4th denied, resets after the window")

    # Sliding window log: rolls continuously, no edge burst.
    clk2 = Clock()
    sw = SlidingWindowLog(limit=2, window=1.0, clock=clk2)
    assert sw.allow() and sw.allow() and not sw.allow()   # 2 then blocked
    clk2.tick(1.01)                              # oldest expires
    assert sw.allow() is True
    print("   sliding window log: exact rolling count; no fixed-window edge burst")

    # Token bucket: burst up to capacity, then throttled to the refill rate.
    clk3 = Clock()
    tb = TokenBucket(capacity=5, refill_per_sec=1.0, clock=clk3)
    assert sum(tb.allow() for _ in range(5)) == 5   # burst of 5
    assert tb.allow() is False                      # bucket empty
    clk3.tick(2.0)                                  # refill 2 tokens
    assert sum(tb.allow() for _ in range(3)) == 2   # only 2 available
    print("   token bucket: allows a burst of 5, then refills at 1/sec")

In [ ]:
def main() -> None:
    print("=" * 70)
    print("SYSTEM DESIGN — rate_limiting.py")
    print("=" * 70)
    print("Four algorithms to cap request rate (fake clock = deterministic tests):")
    demo()
    print("-" * 70)
    print("Lesson: token bucket allows controlled bursts; sliding-window is exact; fixed-window is simplest but bursts at edges.")
    print("All rate_limiting demos passed ✔")


if __name__ == "__main__":
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()